## Natural Language Processing Spring 2026
---
# LoRA Fine-Tuning and DPO (RLHF) on MiniMind

**Objective:** Take a *Supervised Fine-Tuned* (SFT) MiniMind checkpoint and run two of the most common post-training steps used in modern LLM pipelines, end-to-end:

1. **Acquire an SFT-only checkpoint** — the public MiniMind release ships a `full_sft_{hidden_size}.pth` weight (no RLHF applied yet). When that exact file is unavailable for a given configuration, we follow the recipe from `minimind_data_pretrain.ipynb` to *create* one from scratch.
2. **LoRA Fine-Tuning** — adapt the SFT model to a new style/domain by training only a tiny pair of low-rank matrices, exactly as `trainer/train_lora.py` and `model/model_lora.py` do.
3. **DPO (Direct Preference Optimization)** — perform RLHF by directly optimizing the policy on (chosen, rejected) preference pairs against a frozen reference model, mirroring `trainer/train_dpo.py` and `dataset/lm_dataset.py:DPODataset`.
4. **Side-by-Side Comparison** — generate completions from the same prompt with the SFT model, the SFT+LoRA-merged model, and the DPO model, and compare them.

This notebook is a companion to `MiniMindModel.ipynb` (architecture) and `minimind_data_pretrain.ipynb` (data + pre-training + SFT). It re-uses the same micro-architecture (`hidden_size=256`, `num_hidden_layers=2`) so that the entire pipeline still fits in a free Colab session.

---

### Where this fits in the MiniMind training pipeline

| Phase | Notebook | Output checkpoint | Role |
|---|---|---|---|
| 1. Pre-training | `minimind_data_pretrain.ipynb` | `pretrain_model.pth` | Learn language structure |
| 2. SFT          | `minimind_data_pretrain.ipynb` | `sft_model.pth`      | Learn to follow instructions |
| 3a. LoRA        | **this notebook**             | `lora_demo.pth` (+ merged checkpoint) | Lightweight domain/style adaptation |
| 3b. DPO (RLHF)  | **this notebook**             | `dpo_model.pth`      | Align with human preferences |

The official repo file-naming convention (see `README.md`) is `full_sft_{hidden_size}.pth`, `lora_xxx_{hidden_size}.pth`, `dpo_{hidden_size}.pth`; we use shorter demo names below for clarity.

## Step 0: Setup and Imports

### Library Overview

| Import | Role |
|---|---|
| `os`, `json`, `copy` | File system, JSON parsing, deep copies for the frozen reference model |
| `torch`, `torch.nn`, `torch.nn.functional` | Core tensor ops, layers, and loss functions |
| `torch.optim` | `AdamW` optimiser used by every training stage |
| `torch.utils.data.Dataset` / `DataLoader` | Same `Dataset` contract used in the previous notebook |
| `contextlib.nullcontext` | No-op context for `autocast` on CPU |
| `huggingface_hub.hf_hub_download` | Pulls the MiniMind preference dataset (`dpo.jsonl`) from the Hub |
| `datasets.load_dataset` | Streaming JSONL reader for the preference data |


In [ ]:
!pip install -q huggingface_hub datasets

import os
import json
import math
import time
import copy
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset, DataLoader
from contextlib import nullcontext
from datasets import load_dataset, Features, Value
from huggingface_hub import hf_hub_download

## Step 1: Download Datasets and Acquire the SFT Checkpoint

### Datasets we need

| File | Format | Purpose |
|---|---|---|
| `pretrain_t2t_mini.jsonl` | `{"text": "..."}` | Re-create the SFT base by running pre-training (Step 3) |
| `sft_t2t_mini.jsonl` | `{"conversations": [{role, content}, ...]}` | SFT data + LoRA fine-tuning data |
| `dpo.jsonl` | `{"chosen": [...], "rejected": [...]}` | RLHF preference data — see `README.md §IV` and `dataset/lm_dataset.py:DPODataset` |

### Note on the SFT checkpoint

The MiniMind release publishes `full_sft_{hidden_size}.pth` (SFT only — *no* DPO/PPO/GRPO applied) on `jingyaogong/minimind-3-pytorch`. That checkpoint targets the production architecture (`hidden_size=768`, `num_hidden_layers=8`) and **cannot be loaded into the micro 256-dim demo model used here**. Following the convention of the previous notebook, we therefore **recreate** an SFT checkpoint from scratch in Step 3 using the very same `pretrain_t2t_mini.jsonl` → `sft_t2t_mini.jsonl` recipe. In production you would simply run

```python
hf_hub_download(
    repo_id="jingyaogong/minimind-3-pytorch",
    filename=f"full_sft_{lm_config.hidden_size}.pth",
    local_dir="./checkpoints",
)
```

and skip Step 3.

In [ ]:
# Download the three JSONL files we need.
repo_id = "jingyaogong/minimind_dataset"
files_to_download = [
    "pretrain_t2t_mini.jsonl",
    "sft_t2t_mini.jsonl",
    "dpo.jsonl",
]

dataset_dir = "./dataset"
os.makedirs(dataset_dir, exist_ok=True)

print("Downloading datasets from Hugging Face...")
for file_name in files_to_download:
    print(f"Fetching {file_name}...")
    hf_hub_download(
        repo_id=repo_id,
        filename=file_name,
        repo_type="dataset",
        local_dir=dataset_dir,
    )

print("\nAll datasets downloaded to ./dataset.")

In [ ]:
# Quick statistics for the preference dataset (and a peek at one record).
dpo_path = "./dataset/dpo.jsonl"
if os.path.exists(dpo_path):
    count = 0
    first_sample = None
    with open(dpo_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i == 0:
                first_sample = json.loads(line)
            count += 1
    print(f"dpo.jsonl: {count:,} preference pairs")
    print("Example record:")
    print(json.dumps(first_sample, ensure_ascii=False, indent=2)[:600])

## Step 2: MiniMind Architecture (Standalone Recap)

We re-include the same micro-architecture used in `minimind_data_pretrain.ipynb` so this notebook is self-contained. The forward signature `(input_ids, position_embeddings, labels=None) -> (logits, loss)` is identical to the previous notebook so the LoRA / DPO trainers below can plug in directly.

In [ ]:
# 1. Configuration & Normalization
class MiniMindConfig:
    """Hyperparameter container — kept tiny for Colab."""
    def __init__(self):
        self.vocab_size = 6400
        self.hidden_size = 256
        self.num_hidden_layers = 2
        self.num_attention_heads = 8
        self.num_key_value_heads = 4
        self.head_dim = self.hidden_size // self.num_attention_heads
        self.intermediate_size = 1024
        self.max_position_embeddings = 2048
        self.rms_norm_eps = 1e-6
        self.dropout = 0.0
        self.tie_word_embeddings = True

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        normed = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (self.weight * normed.float()).type_as(x)

# 2. Rotary Position Embeddings
def precompute_freqs_cis(dim: int, end: int, rope_base: float = 1e6):
    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()
    freqs_cos = torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1)
    freqs_sin = torch.cat([torch.sin(freqs), torch.sin(freqs)], dim=-1)
    return freqs_cos, freqs_sin

def apply_rotary_pos_emb(q, k, cos, sin):
    def rotate_half(x):
        half_dim = x.shape[-1] // 2
        return torch.cat((-x[..., half_dim:], x[..., :half_dim]), dim=-1)
    cos_u = cos.unsqueeze(0).unsqueeze(2)
    sin_u = sin.unsqueeze(0).unsqueeze(2)
    return (q * cos_u) + (rotate_half(q) * sin_u), (k * cos_u) + (rotate_half(k) * sin_u)

# 3. Attention & FFN
def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    bs, slen, num_kv_heads, head_dim = x.shape
    if n_rep == 1:
        return x
    return x[:, :, :, None, :].expand(bs, slen, num_kv_heads, n_rep, head_dim).reshape(bs, slen, num_kv_heads * n_rep, head_dim)

class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_local_heads = config.num_attention_heads
        self.n_local_kv_heads = config.num_key_value_heads
        self.n_rep = self.n_local_heads // self.n_local_kv_heads
        self.head_dim = config.head_dim
        self.q_proj = nn.Linear(config.hidden_size, self.n_local_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, self.n_local_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, self.n_local_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.n_local_heads * self.head_dim, config.hidden_size, bias=False)
        self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
    def forward(self, x, position_embeddings):
        bsz, seq_len, _ = x.shape
        xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        xq = xq.view(bsz, seq_len, self.n_local_heads, self.head_dim)
        xk = xk.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xv = xv.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xq, xk = self.q_norm(xq), self.k_norm(xk)
        cos, sin = position_embeddings
        xq, xk = apply_rotary_pos_emb(xq, xk, cos, sin)
        xq = xq.transpose(1, 2)
        xk = repeat_kv(xk, self.n_rep).transpose(1, 2)
        xv = repeat_kv(xv, self.n_rep).transpose(1, 2)
        scores = (xq @ xk.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.full((seq_len, seq_len), float("-inf"), device=scores.device).triu(1)
        scores = scores + mask
        attention_output = F.softmax(scores, dim=-1) @ xv
        attention_output = attention_output.transpose(1, 2).contiguous().reshape(bsz, seq_len, -1)
        return self.o_proj(attention_output)

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj   = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class MiniMindBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.self_attn = Attention(config)
        self.input_layernorm = RMSNorm(config.hidden_size)
        self.post_attention_layernorm = RMSNorm(config.hidden_size)
        self.mlp = FeedForward(config)
    def forward(self, x, position_embeddings):
        x = x + self.self_attn(self.input_layernorm(x), position_embeddings)
        x = x + self.mlp(self.post_attention_layernorm(x))
        return x

class MiniMindForCausalLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([MiniMindBlock(config) for _ in range(config.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        if config.tie_word_embeddings:
            self.embed_tokens.weight = self.lm_head.weight
    def forward(self, input_ids, position_embeddings, labels=None):
        x = self.embed_tokens(input_ids)
        for layer in self.layers:
            x = layer(x, position_embeddings)
        x = self.norm(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), ignore_index=-100)
        return logits, loss

## Step 3: Build the SFT (Non-RLHF) Checkpoint

This step is a condensed re-run of `minimind_data_pretrain.ipynb`: pre-train on `pretrain_t2t_mini.jsonl`, then SFT on `sft_t2t_mini.jsonl`, and save `./checkpoints/sft_model.pth`. That file plays the role of the official `full_sft_{hidden_size}.pth` for the rest of this notebook.

We keep the data subsets small so the whole notebook fits in a Colab session — increase the `select(range(N))` calls below for better quality.

In [ ]:
# Datasets — same definitions as in minimind_data_pretrain.ipynb.

class PretrainDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_length=512):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = load_dataset('json', data_files=data_path, split='train')
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, index):
        sample = self.samples[index]
        tokens = self.tokenizer(str(sample['text']), add_special_tokens=False, max_length=self.max_length - 2, truncation=True).input_ids
        tokens = [self.tokenizer.bos_token_id] + tokens + [self.tokenizer.eos_token_id]
        input_ids = tokens + [self.tokenizer.pad_token_id] * (self.max_length - len(tokens))
        input_ids = torch.tensor(input_ids, dtype=torch.long)
        labels = input_ids.clone()
        labels[input_ids == self.tokenizer.pad_token_id] = -100
        return input_ids, labels

class SFTDataset(Dataset):
    """Same SFT logic as in minimind_data_pretrain.ipynb — masks every non-assistant token."""
    def __init__(self, jsonl_path, tokenizer, max_length=1024):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        features = Features({'conversations': [{'role': Value('string'), 'content': Value('string'), 'reasoning_content': Value('string'), 'tools': Value('string'), 'tool_calls': Value('string')}]})
        self.samples = load_dataset('json', data_files=jsonl_path, split='train', features=features)
    def __len__(self):
        return len(self.samples)
    def _encode(self, text):
        return self.tokenizer(text, add_special_tokens=False, max_length=self.max_length, truncation=False).input_ids
    def __getitem__(self, index):
        sample = self.samples[index]
        conversations = sample['conversations']
        input_ids = [self.tokenizer.bos_token_id]
        labels = [-100]
        for turn in conversations:
            role = (turn.get('role') or 'user')
            content = str(turn.get('content') or '')
            header_ids = self._encode(f"<{role}>\n")
            content_ids = self._encode(content)
            input_ids += header_ids + content_ids
            if role == 'assistant':
                labels += [-100] * len(header_ids) + list(content_ids)
            else:
                labels += [-100] * (len(header_ids) + len(content_ids))
        input_ids.append(self.tokenizer.eos_token_id)
        if len(labels) and labels[-1] != -100:
            labels.append(self.tokenizer.eos_token_id)
        else:
            labels.append(-100)
        input_ids = input_ids[:self.max_length]
        labels = labels[:self.max_length]
        pad_len = self.max_length - len(input_ids)
        input_ids += [self.tokenizer.pad_token_id] * pad_len
        labels += [-100] * pad_len
        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

class MockTokenizer:
    """Deterministic char-based tokenizer — same one used in the previous notebook."""
    bos_token_id = 1
    eos_token_id = 2
    pad_token_id = 0
    bos_token = '<s>'
    eos_token = '</s>'
    def __call__(self, text, add_special_tokens=False, max_length=512, truncation=True):
        class Output:
            input_ids = [(ord(c) % 6300) + 10 for c in text][:max_length]
        return Output()
    def decode(self, token_ids):
        return "".join([chr(t - 10) if 32 <= (t - 10) <= 126 else "*" for t in token_ids])

### Training loop with checkpointing

Same techniques as the previous notebook (mixed precision via `torch.amp.autocast`, `GradScaler`, gradient accumulation and clipping). We save once at the end of each phase.

In [ ]:
def train_phase(model, loader, optimizer, scaler, args, pos_embs, phase_name, save_path, epochs):
    """Identical recipe to `train_phase_demo` in minimind_data_pretrain.ipynb."""
    device_type = "cuda" if "cuda" in args.device else "cpu"
    autocast_ctx = nullcontext() if device_type == "cpu" else torch.amp.autocast('cuda', dtype=torch.float16)
    model.train()
    print(f"\n>>> Starting Phase: {phase_name} <<<")
    for epoch in range(1, epochs + 1):
        last_step = 0
        for step, (input_ids, labels) in enumerate(loader, start=1):
            last_step = step
            input_ids, labels = input_ids.to(args.device), labels.to(args.device)
            with autocast_ctx:
                logits, loss = model(input_ids, position_embeddings=pos_embs, labels=labels)
                loss = loss / args.accumulation_steps
            scaler.scale(loss).backward()
            if step % args.accumulation_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            if step % args.log_interval == 0:
                print(f"[{phase_name}] Epoch {epoch}/{epochs} Step {step}: Loss = {loss.item() * args.accumulation_steps:.4f}")
        if last_step and last_step % args.accumulation_steps != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True)
    print(f"Saving {phase_name} checkpoint to {save_path}...")
    torch.save(model.state_dict(), save_path)
    print("Saved.")

In [ ]:
class TrainArgs:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    epochs = 3
    learning_rate = 5e-4
    sft_learning_rate = 5e-5
    accumulation_steps = 2
    grad_clip = 1.0
    log_interval = 5
    max_seq_len = 32

args = TrainArgs()
config = MiniMindConfig()
os.makedirs("./checkpoints", exist_ok=True)

tokenizer = MockTokenizer()
model = MiniMindForCausalLM(config).to(args.device)
pos_embs = precompute_freqs_cis(config.head_dim, args.max_seq_len)
pos_embs = (pos_embs[0].to(args.device), pos_embs[1].to(args.device))

# 1. Pre-training (small subset).
ds_pretrain = PretrainDataset("./dataset/pretrain_t2t_mini.jsonl", tokenizer, max_length=args.max_seq_len)
ds_pretrain.samples = ds_pretrain.samples.select(range(20))
loader_pretrain = DataLoader(ds_pretrain, batch_size=4, shuffle=True)
opt_pt = optim.AdamW(model.parameters(), lr=args.learning_rate)
scaler_pt = torch.amp.GradScaler("cuda", enabled=("cuda" in args.device))
train_phase(model, loader_pretrain, opt_pt, scaler_pt, args, pos_embs,
            phase_name="Pre-training", save_path="./checkpoints/pretrain_model.pth", epochs=args.epochs)

# 2. SFT (this is the 'full_sft' equivalent — no RLHF applied yet).
ds_sft = SFTDataset("./dataset/sft_t2t_mini.jsonl", tokenizer, max_length=args.max_seq_len)
ds_sft.samples = ds_sft.samples.select(range(2000))
loader_sft = DataLoader(ds_sft, batch_size=4, shuffle=True)
opt_sft = optim.AdamW(model.parameters(), lr=args.sft_learning_rate)
scaler_sft = torch.amp.GradScaler("cuda", enabled=("cuda" in args.device))
train_phase(model, loader_sft, opt_sft, scaler_sft, args, pos_embs,
            phase_name="Supervised Fine-Tuning", save_path="./checkpoints/sft_model.pth", epochs=args.epochs)

print("\nSFT (non-RLHF) checkpoint ready: ./checkpoints/sft_model.pth")

## Step 4: LoRA — Low-Rank Adaptation Module

The implementation below mirrors `model/model_lora.py` in the repo verbatim. The idea, from [Hu et al. 2021](https://arxiv.org/abs/2106.09685), is to **freeze** every weight of a pre-trained model and train only a tiny update of the form

$$W' = W + \Delta W = W + B A,\quad A \in \mathbb{R}^{r \times d_{\text{in}}},\; B \in \mathbb{R}^{d_{\text{out}} \times r}$$

where the rank $r$ (typically 8–64) is dramatically smaller than $d_{\text{in}}, d_{\text{out}}$. Two subtle but important details from the reference code:

* **Initialisation:** `A` is sampled from $\mathcal{N}(0, 0.02^2)$ while `B` starts at **zero**, so at training step 0 we have $\Delta W = BA = 0$ — the LoRA-augmented model is exactly identical to the frozen base model. Without this, training would start from a random perturbation and immediately destroy SFT behaviour.
* **Square-only filter:** `apply_lora` skips non-square `nn.Linear` modules (those with `weight.shape[0] != weight.shape[1]`). In MiniMind this means LoRA is added to `q_proj`, `o_proj`, `lm_head`, and the FFN's square-only projections; `k_proj`/`v_proj` (which are smaller because of GQA) and the rectangular FFN projections are skipped. This matches the reference behaviour exactly.

### `save_lora` / `load_lora` / `merge_lora`

* `save_lora` walks the model, picks out every `module.lora` it finds, and writes only the `A`/`B` weights — the resulting file is therefore tiny (a few MB instead of the full model size).
* `load_lora` reverses this: it scans the saved state dict and routes each `A`/`B` tensor back to the matching `module.lora`.
* `merge_lora` collapses the LoRA into the base weights via $W \leftarrow W + BA$ and saves a *single* full checkpoint — useful when you want to deploy the LoRA-adapted model without keeping the LoRA wrapper at inference time.

In [ ]:
# ---- model/model_lora.py (verbatim semantics) ---------------------------
class LoRA(nn.Module):
    def __init__(self, in_features, out_features, rank):
        super().__init__()
        self.rank = rank
        self.A = nn.Linear(in_features, rank, bias=False)   # low-rank A
        self.B = nn.Linear(rank, out_features, bias=False)  # low-rank B
        self.A.weight.data.normal_(mean=0.0, std=0.02)      # Gaussian init for A
        self.B.weight.data.zero_()                          # Zero init for B → ΔW = 0 at step 0
    def forward(self, x):
        return self.B(self.A(x))

def apply_lora(model, rank=8):
    """Attach a LoRA branch to every square nn.Linear in the model.
    Mirrors model/model_lora.py:apply_lora — only the device lookup is adapted to
    the demo model (the production model exposes a `.device` attribute).
    """
    device = next(model.parameters()).device
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and module.weight.shape[0] == module.weight.shape[1]:
            lora = LoRA(module.weight.shape[0], module.weight.shape[1], rank=rank).to(device)
            setattr(module, "lora", lora)
            original_forward = module.forward
            # Late-binding pitfall: capture both layers as default args.
            def forward_with_lora(x, layer1=original_forward, layer2=lora):
                return layer1(x) + layer2(x)
            module.forward = forward_with_lora

def save_lora(model, path):
    raw_model = getattr(model, '_orig_mod', model)
    state_dict = {}
    for name, module in raw_model.named_modules():
        if hasattr(module, 'lora'):
            clean_name = name[7:] if name.startswith("module.") else name
            lora_state = {f'{clean_name}.lora.{k}': v.cpu().half() for k, v in module.lora.state_dict().items()}
            state_dict.update(lora_state)
    torch.save(state_dict, path)

def load_lora(model, path):
    device = next(model.parameters()).device
    state_dict = torch.load(path, map_location=device)
    state_dict = {(k[7:] if k.startswith('module.') else k): v for k, v in state_dict.items()}
    for name, module in model.named_modules():
        if hasattr(module, 'lora'):
            lora_state = {k.replace(f'{name}.lora.', ''): v for k, v in state_dict.items() if f'{name}.lora.' in k}
            module.lora.load_state_dict(lora_state)

def merge_lora(model, lora_path, save_path):
    """Merge LoRA into base weights and save a single checkpoint (W ← W + BA)."""
    load_lora(model, lora_path)
    raw_model = getattr(model, '_orig_mod', model)
    state_dict = {k: v.cpu().half() for k, v in raw_model.state_dict().items() if '.lora.' not in k}
    for name, module in raw_model.named_modules():
        if isinstance(module, nn.Linear) and '.lora.' not in name:
            state_dict[f'{name}.weight'] = module.weight.data.clone().cpu().half()
            if hasattr(module, 'lora'):
                state_dict[f'{name}.weight'] += (module.lora.B.weight.data @ module.lora.A.weight.data).cpu().half()
    torch.save(state_dict, save_path)

## Step 5: LoRA Fine-Tuning Loop

This mirrors `trainer/train_lora.py`. The recipe is:

1. Load the SFT checkpoint into a fresh model.
2. Call `apply_lora` to attach LoRA branches.
3. Set `requires_grad = True` only on parameters whose name contains `'lora'`; freeze everything else.
4. Build an `AdamW` over the LoRA parameters only.
5. Train on SFT-format data (the same `SFTDataset` used in Step 3 — in production this is `lora_medical.jsonl`, `lora_identity.jsonl`, etc.).
6. Save with `save_lora` (LoRA-only weights) and optionally `merge_lora` for inference.

Notice that the loss is the standard cross-entropy *on top of the SFT objective* — LoRA does not change the loss, only the parameter set being optimised.

> The original `train_lora.py` also adds `res.aux_loss` (used by the MoE variant). Our dense demo model has no MoE, so there is no auxiliary loss term to add.

In [ ]:
# 1. Fresh model + load SFT (non-RLHF) weights.
model_lora = MiniMindForCausalLM(config).to(args.device)
model_lora.load_state_dict(torch.load("./checkpoints/sft_model.pth", weights_only=True, map_location=args.device))

# 2. Attach LoRA branches (rank=8 — same default as MiniMind's medical/identity LoRAs).
apply_lora(model_lora, rank=8)

# 3. Freeze everything except LoRA. Mirrors trainer/train_lora.py lines 138–144.
lora_params = []
for name, param in model_lora.named_parameters():
    if 'lora' in name:
        param.requires_grad = True
        lora_params.append(param)
    else:
        param.requires_grad = False

total_params = sum(p.numel() for p in model_lora.parameters())
lora_count   = sum(p.numel() for p in lora_params)
print(f"Total params : {total_params/1e6:.3f} M")
print(f"LoRA params  : {lora_count/1e6:.3f} M  ({lora_count/total_params*100:.2f}%)")

# 4. SFT-style loader (same dataset object used to build sft_model.pth).
loader_lora = DataLoader(ds_sft, batch_size=4, shuffle=True)

# 5. AdamW over the LoRA parameters only.
opt_lora = optim.AdamW(lora_params, lr=1e-4)
scaler_lora = torch.amp.GradScaler("cuda", enabled=("cuda" in args.device))

# 6. Reuse the standard training loop. Gradient clipping must operate on the
#    LoRA params only, exactly like trainer/train_lora.py:44.
device_type = "cuda" if "cuda" in args.device else "cpu"
autocast_ctx = nullcontext() if device_type == "cpu" else torch.amp.autocast('cuda', dtype=torch.float16)
model_lora.train()
print("\n>>> Starting Phase: LoRA Fine-Tuning <<<")
for epoch in range(1, args.epochs + 1):
    last_step = 0
    for step, (input_ids, labels) in enumerate(loader_lora, start=1):
        last_step = step
        input_ids, labels = input_ids.to(args.device), labels.to(args.device)
        with autocast_ctx:
            logits, loss = model_lora(input_ids, position_embeddings=pos_embs, labels=labels)
            loss = loss / args.accumulation_steps
        scaler_lora.scale(loss).backward()
        if step % args.accumulation_steps == 0:
            scaler_lora.unscale_(opt_lora)
            torch.nn.utils.clip_grad_norm_(lora_params, args.grad_clip)
            scaler_lora.step(opt_lora); scaler_lora.update()
            opt_lora.zero_grad(set_to_none=True)
        if step % args.log_interval == 0:
            print(f"[LoRA] Epoch {epoch}/{args.epochs} Step {step}: Loss = {loss.item()*args.accumulation_steps:.4f}")
    if last_step and last_step % args.accumulation_steps != 0:
        scaler_lora.unscale_(opt_lora)
        torch.nn.utils.clip_grad_norm_(lora_params, args.grad_clip)
        scaler_lora.step(opt_lora); scaler_lora.update()
        opt_lora.zero_grad(set_to_none=True)

# 7. Save LoRA-only weights and a merged full checkpoint for evaluation.
save_lora(model_lora, "./checkpoints/lora_demo.pth")
print("Saved LoRA-only weights to ./checkpoints/lora_demo.pth")

# Build a *fresh* base model + merge LoRA into it, leaving model_lora untouched.
merge_base = MiniMindForCausalLM(config).to(args.device)
merge_base.load_state_dict(torch.load("./checkpoints/sft_model.pth", weights_only=True, map_location=args.device))
apply_lora(merge_base, rank=8)
merge_lora(merge_base, "./checkpoints/lora_demo.pth", "./checkpoints/sft_lora_merged.pth")
print("Saved merged SFT+LoRA checkpoint to ./checkpoints/sft_lora_merged.pth")

## Step 6: DPO Preference Dataset

The implementation below is the demo equivalent of `dataset/lm_dataset.py:DPODataset`. Each record contains two role-formatted conversations — `chosen` (preferred response) and `rejected` (worse response) — that share the same user prompt. We need three tensors per side:

| Field | Shape | What it is |
|---|---|---|
| `x_chosen`, `x_rejected` | `(seq_len-1,)` | Input tokens (input shifted left, suitable for autoregressive LM) |
| `y_chosen`, `y_rejected` | `(seq_len-1,)` | Target tokens (input shifted right) |
| `mask_chosen`, `mask_rejected` | `(seq_len-1,)` | 1 where the target falls inside an *assistant* turn, 0 otherwise — used by the DPO loss to ignore prompt tokens |

The original code uses `tokenizer.apply_chat_template` and scans for `<bos>assistant\n ... <eos>\n` sentinels. Because our `MockTokenizer` lacks chat templating, we build the chat string manually with the same `<role>\n` markers as the SFT data pipeline and mark every assistant span as supervised — the **exact same convention** used by `SFTDataset` in Step 3.

In [ ]:
class DPODataset(Dataset):
    """Demo DPO dataset — same outputs as dataset/lm_dataset.py:DPODataset."""
    def __init__(self, jsonl_path, tokenizer, max_length=128):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = load_dataset('json', data_files=jsonl_path, split='train')

    def __len__(self):
        return len(self.samples)

    def _encode_pair(self, conversation):
        """Tokenise one (chosen|rejected) conversation and produce input_ids + loss mask.
        loss_mask[i] == 1 iff token i is part of an assistant turn (the only positions DPO supervises).
        """
        input_ids = [self.tokenizer.bos_token_id]
        loss_mask = [0]
        for turn in conversation:
            role = (turn.get('role') or 'user')
            content = str(turn.get('content') or '')
            header_ids  = self.tokenizer(f"<{role}>\n", add_special_tokens=False, max_length=self.max_length, truncation=True).input_ids
            content_ids = self.tokenizer(content,        add_special_tokens=False, max_length=self.max_length, truncation=True).input_ids
            input_ids += header_ids + content_ids
            if role == 'assistant':
                loss_mask += [0] * len(header_ids) + [1] * len(content_ids)
            else:
                loss_mask += [0] * (len(header_ids) + len(content_ids))
        input_ids.append(self.tokenizer.eos_token_id)
        loss_mask.append(1 if (len(loss_mask) and loss_mask[-1] == 1) else 0)
        # Truncate / pad.
        input_ids = input_ids[:self.max_length]
        loss_mask = loss_mask[:self.max_length]
        pad_len = self.max_length - len(input_ids)
        input_ids += [self.tokenizer.pad_token_id] * pad_len
        loss_mask += [0] * pad_len
        return input_ids, loss_mask

    def __getitem__(self, index):
        sample = self.samples[index]
        chosen_ids,   chosen_mask   = self._encode_pair(sample['chosen'])
        rejected_ids, rejected_mask = self._encode_pair(sample['rejected'])
        # Same shift as dataset/lm_dataset.py:DPODataset.__getitem__.
        x_chosen   = torch.tensor(chosen_ids[:-1],    dtype=torch.long)
        y_chosen   = torch.tensor(chosen_ids[1:],     dtype=torch.long)
        m_chosen   = torch.tensor(chosen_mask[1:],    dtype=torch.long)
        x_rejected = torch.tensor(rejected_ids[:-1],  dtype=torch.long)
        y_rejected = torch.tensor(rejected_ids[1:],   dtype=torch.long)
        m_rejected = torch.tensor(rejected_mask[1:],  dtype=torch.long)
        return {
            'x_chosen': x_chosen, 'y_chosen': y_chosen, 'mask_chosen': m_chosen,
            'x_rejected': x_rejected, 'y_rejected': y_rejected, 'mask_rejected': m_rejected,
        }

# Build the dataset over the first slice of dpo.jsonl so the demo stays fast.
dpo_seq_len = 32
ds_dpo = DPODataset("./dataset/dpo.jsonl", tokenizer, max_length=dpo_seq_len)
ds_dpo.samples = ds_dpo.samples.select(range(min(200, len(ds_dpo.samples))))
loader_dpo = DataLoader(ds_dpo, batch_size=4, shuffle=True)
print(f"DPO dataset: {len(ds_dpo)} preference pairs, max_seq_len={dpo_seq_len}")

## Step 7: DPO Loss and Training Loop

### The DPO loss

Following [Rafailov et al. 2023](https://arxiv.org/abs/2305.18290), Direct Preference Optimization replaces the usual reward-model-+-PPO setup with a single supervised loss derived analytically from the same KL-constrained RL objective:

$$\mathcal{L}_{\text{DPO}}(\pi_\theta;\pi_{\text{ref}}) = -\,\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}}\Bigg[\log\sigma\!\Big(\beta\,\big[\log\tfrac{\pi_\theta(y_w\,|\,x)}{\pi_{\text{ref}}(y_w\,|\,x)} - \log\tfrac{\pi_\theta(y_l\,|\,x)}{\pi_{\text{ref}}(y_l\,|\,x)}\big]\Big)\Bigg]$$

where $y_w$ is the chosen (winning) response, $y_l$ is the rejected (losing) one, $\pi_\theta$ is the policy we train, $\pi_{\text{ref}}$ is a frozen copy of the SFT model, and $\beta$ controls how much we trust the preference data versus the SFT prior.

The implementation in `trainer/train_dpo.py` is split into two helpers, both reproduced verbatim below:

* `logits_to_log_probs(logits, labels)` — gathers the per-token log-probabilities the model assigned to the gold label sequence. Shape: `(batch, seq_len)`.
* `dpo_loss(ref_log_probs, policy_log_probs, mask, beta)` — sums log-probs over the assistant span (`mask`), splits the doubled batch into the chosen / rejected halves (the DPO trainer concatenates them along dim 0), and applies the formula above.

### The reference model

The reference policy $\pi_{\text{ref}}$ is a **frozen, untrained deep-copy of the SFT model**. We never call `.backward()` on it, and `requires_grad_(False)` plus `eval()` ensure no gradient or BatchNorm/Dropout state leaks in.

In [ ]:
# ---- trainer/train_dpo.py (verbatim) ----------------------------------
def logits_to_log_probs(logits, labels):
    log_probs = F.log_softmax(logits, dim=2)
    return torch.gather(log_probs, dim=2, index=labels.unsqueeze(2)).squeeze(-1)

def dpo_loss(ref_log_probs, policy_log_probs, mask, beta):
    # Sum log-probabilities only over assistant tokens.
    ref_log_probs    = (ref_log_probs    * mask).sum(dim=1)
    policy_log_probs = (policy_log_probs * mask).sum(dim=1)
    # The trainer below stacks chosen + rejected along dim 0; split them here.
    bsz = ref_log_probs.shape[0]
    chosen_ref,    reject_ref    = ref_log_probs[:bsz // 2],    ref_log_probs[bsz // 2:]
    chosen_policy, reject_policy = policy_log_probs[:bsz // 2], policy_log_probs[bsz // 2:]
    pi_logratios  = chosen_policy - reject_policy
    ref_logratios = chosen_ref    - reject_ref
    logits = pi_logratios - ref_logratios
    return -F.logsigmoid(beta * logits).mean()

### DPO trainer

The structure mirrors `trainer/train_dpo.py:train_epoch`:

1. Concatenate `chosen` and `rejected` along the batch dimension so we can do **a single forward pass** through the policy and a single forward pass through the reference model.
2. Run `model(x)` to get `policy_logits`, then `logits_to_log_probs(policy_logits, y)` to get per-token log-probs of the gold targets.
3. Run `ref_model(x)` under `torch.no_grad()` and compute the same log-probs (the reference is frozen).
4. Apply `dpo_loss` with `beta=0.15` (the default in `train_dpo.py`), divide by `accumulation_steps`, backprop, optimiser step.

> The original repo's `outputs.logits` / `outputs.aux_loss` come from the production `MiniMindForCausalLM` which returns a HuggingFace-style dataclass. Our micro model returns the `(logits, loss)` tuple, so we simply unpack it. There is no `aux_loss` term in this dense demo model.

**Learning rate:** DPO is *very* sensitive to LR — `train_dpo.py` defaults to `4e-8`. We bump it slightly here so a few hundred steps move the loss visibly on the small subset; in production stick to ≤5e-8.

In [ ]:
# Build the DPO policy + reference model from the SFT (non-RLHF) checkpoint.
model_dpo = MiniMindForCausalLM(config).to(args.device)
model_dpo.load_state_dict(torch.load("./checkpoints/sft_model.pth", weights_only=True, map_location=args.device))

ref_model = MiniMindForCausalLM(config).to(args.device)
ref_model.load_state_dict(torch.load("./checkpoints/sft_model.pth", weights_only=True, map_location=args.device))
ref_model.eval()
ref_model.requires_grad_(False)

# Position embeddings for the (potentially shorter) DPO sequence length.
pos_embs_dpo = precompute_freqs_cis(config.head_dim, dpo_seq_len)
pos_embs_dpo = (pos_embs_dpo[0].to(args.device), pos_embs_dpo[1].to(args.device))

opt_dpo    = optim.AdamW(model_dpo.parameters(), lr=1e-6)
scaler_dpo = torch.amp.GradScaler("cuda", enabled=("cuda" in args.device))
beta = 0.15
device_type  = "cuda" if "cuda" in args.device else "cpu"
autocast_ctx = nullcontext() if device_type == "cpu" else torch.amp.autocast('cuda', dtype=torch.float16)

model_dpo.train()
print("\n>>> Starting Phase: DPO (RLHF) <<<")
for epoch in range(1, args.epochs + 1):
    last_step = 0
    for step, batch in enumerate(loader_dpo, start=1):
        last_step = step
        # Stack chosen + rejected along dim 0 — same trick as train_dpo.py:64-66.
        x = torch.cat([batch['x_chosen'],    batch['x_rejected']],    dim=0).to(args.device)
        y = torch.cat([batch['y_chosen'],    batch['y_rejected']],    dim=0).to(args.device)
        m = torch.cat([batch['mask_chosen'], batch['mask_rejected']], dim=0).to(args.device)
        # Position embeddings only need to cover the actual sequence length we feed in.
        cur_len = x.size(1)
        pe = (pos_embs_dpo[0][:cur_len], pos_embs_dpo[1][:cur_len])
        with autocast_ctx:
            with torch.no_grad():
                ref_logits, _ = ref_model(x, position_embeddings=pe)
                ref_log_probs = logits_to_log_probs(ref_logits, y)
            policy_logits, _ = model_dpo(x, position_embeddings=pe)
            policy_log_probs = logits_to_log_probs(policy_logits, y)
            loss = dpo_loss(ref_log_probs, policy_log_probs, m, beta=beta)
            loss = loss / args.accumulation_steps
        scaler_dpo.scale(loss).backward()
        if step % args.accumulation_steps == 0:
            scaler_dpo.unscale_(opt_dpo)
            torch.nn.utils.clip_grad_norm_(model_dpo.parameters(), args.grad_clip)
            scaler_dpo.step(opt_dpo); scaler_dpo.update()
            opt_dpo.zero_grad(set_to_none=True)
        if step % args.log_interval == 0:
            print(f"[DPO] Epoch {epoch}/{args.epochs} Step {step}: dpo_loss = {loss.item()*args.accumulation_steps:.4f}")
    if last_step and last_step % args.accumulation_steps != 0:
        scaler_dpo.unscale_(opt_dpo)
        torch.nn.utils.clip_grad_norm_(model_dpo.parameters(), args.grad_clip)
        scaler_dpo.step(opt_dpo); scaler_dpo.update()
        opt_dpo.zero_grad(set_to_none=True)

torch.save(model_dpo.state_dict(), "./checkpoints/dpo_model.pth")
print("Saved DPO checkpoint to ./checkpoints/dpo_model.pth")

## Step 8: Side-by-Side Comparison — SFT vs SFT+LoRA vs DPO

We now have three checkpoints written to disk:

| Stage | File | What it is |
|---|---|---|
| 1. SFT (non-RLHF) | `./checkpoints/sft_model.pth` | The starting point — instruction-tuned, no preference learning |
| 2. SFT + LoRA (merged) | `./checkpoints/sft_lora_merged.pth` | SFT + low-rank adaptation merged into the base weights |
| 3. DPO | `./checkpoints/dpo_model.pth` | SFT after RLHF with `dpo.jsonl` |

We load each into its own `MiniMindForCausalLM` instance and feed them all the same prompt with greedy decoding (same approach as `minimind_data_pretrain.ipynb` Step 7).

> Reminder: the demo uses a 256-dim, 2-layer model + a mock ASCII tokenizer + a small subset of every dataset. The text outputs will look like noise. The point is to show that **the same generation function**, fed the **same prompt** and using **three different checkpoints produced by three different post-training procedures**, returns three different outputs. In a full run you would observe the well-known qualitative shifts (LoRA → domain-style replies; DPO → preference-aligned replies).

In [ ]:
def generate_text(model, tokenizer, prompt, max_new_tokens=15):
    model.eval()
    input_ids = torch.tensor([tokenizer(prompt, max_length=args.max_seq_len).input_ids], dtype=torch.long).to(args.device)
    pe_full = precompute_freqs_cis(config.head_dim, input_ids.size(1) + max_new_tokens)
    pe_full = (pe_full[0].to(args.device), pe_full[1].to(args.device))
    with torch.no_grad():
        for _ in range(max_new_tokens):
            cur_len = input_ids.size(1)
            cur_pe = (pe_full[0][:cur_len], pe_full[1][:cur_len])
            logits, _ = model(input_ids, position_embeddings=cur_pe)
            next_token = torch.argmax(logits[:, -1, :], dim=-1).unsqueeze(1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
    return tokenizer.decode(input_ids[0].cpu().tolist())

# Load all three model variants.
model_sft_eval = MiniMindForCausalLM(config).to(args.device)
model_sft_eval.load_state_dict(torch.load("./checkpoints/sft_model.pth", weights_only=True, map_location=args.device))

model_lora_eval = MiniMindForCausalLM(config).to(args.device)
model_lora_eval.load_state_dict(torch.load("./checkpoints/sft_lora_merged.pth", weights_only=True, map_location=args.device))

model_dpo_eval = MiniMindForCausalLM(config).to(args.device)
model_dpo_eval.load_state_dict(torch.load("./checkpoints/dpo_model.pth", weights_only=True, map_location=args.device))

test_prompt = "Hello Assistant, can you help me?"
print(f"[Prompt]: {test_prompt}\n")

out_sft  = generate_text(model_sft_eval,  tokenizer, test_prompt, max_new_tokens=10)
out_lora = generate_text(model_lora_eval, tokenizer, test_prompt, max_new_tokens=10)
out_dpo  = generate_text(model_dpo_eval,  tokenizer, test_prompt, max_new_tokens=10)

print("=== STAGE-BY-STAGE COMPARISON ===")
print("1. SFT (no RLHF) Output:")
print(f"> {out_sft}\n")
print("2. SFT + LoRA (merged) Output:")
print(f"> {out_lora}\n")
print("3. DPO (RLHF) Output:")
print(f"> {out_dpo}\n")
print("=================================")

### What to look for in a real run

* **SFT vs LoRA-merged:** the LoRA-merged model should reflect the style/domain of the LoRA adaptation data. With `lora_medical.jsonl` you would see domain-specific vocabulary in the response; with `lora_identity.jsonl` you would see the model claim its custom identity.
* **SFT vs DPO:** DPO-trained models tend to be more concise, refuse unsafe requests more cleanly, and pick the response style preferred by the annotators in `dpo.jsonl`. Because $\beta$ is small and the reference model regularises toward the SFT distribution, the change in surface form is usually subtle on individual prompts but compounds across an evaluation set.
* **Catastrophic forgetting:** if the DPO learning rate is too high (>5e-8 for the production model) you will see the DPO model drift away from coherent text and hallucinate more — which is why `train_dpo.py` defaults to `4e-8`.

For full evaluation in production, see `eval_llm.py`, which loads `full_sft`, `dpo`, etc. and runs the same prompt suite through each weight.
